# 06-5. UDP 통신 예제

## Goal

- 데이터그램 하나가 메시지 경계 하나임을 확인합니다.
- 손상된 데이터그램을 다른 메시지와 분리해 처리합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

합성 데이터그램 목록을 사용하므로 네트워크 송수신은 발생하지 않습니다.


## Steps

### 데이터그램 단위 검증

각 데이터그램을 독립적으로 디코딩하고 크기·인코딩 오류를 구조화합니다.


In [1]:
MAX_DATAGRAM = 32


def parse_datagram(payload: bytes) -> dict:
    if len(payload) > MAX_DATAGRAM:
        return {"ok": False, "reason": "too_large", "size": len(payload)}
    try:
        text = payload.decode("utf-8")
    except UnicodeDecodeError:
        return {"ok": False, "reason": "invalid_utf8", "size": len(payload)}
    return {"ok": True, "text": text, "size": len(payload)}


datagrams = [b"alpha", "한글".encode("utf-8"), b"\xff\xfe", b"x" * 40]
results = [parse_datagram(item) for item in datagrams]
for result in results:
    print(result)


{'ok': True, 'text': 'alpha', 'size': 5}
{'ok': True, 'text': '한글', 'size': 6}
{'ok': False, 'reason': 'invalid_utf8', 'size': 2}
{'ok': False, 'reason': 'too_large', 'size': 40}


## Checks

한 메시지의 실패가 다른 메시지의 결과를 지우지 않는지 확인합니다.


In [2]:
assert [item["ok"] for item in results] == [True, True, False, False]
assert results[2]["reason"] == "invalid_utf8"
assert results[3]["reason"] == "too_large"
print("데이터그램 검사 통과")


데이터그램 검사 통과


## Next Steps

실제 UDP에서는 응답 손실·중복·순서 변경 가능성을 애플리케이션 규칙으로 다뤄야 합니다.
